# 1.5 - How ML Finds Patterns: The 3 Paradigms

**Module 1 - Math Prerequisites** | Netsetos GenAI Engineering

This notebook runs the four demos behind the lesson's argument that every ML system reduces to *choose a paradigm, choose a loss, then check whether the pattern generalizes*. You'll train a supervised classifier, cluster unlabeled data, watch L1/L2 regularization split real features from noise, and use k-fold to expose how much a single train/test split can lie.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Every demo in this notebook uses seeded random data so the printed numbers are reproducible run-to-run. This cell imports NumPy and pins the global seed; the individual sklearn imports come inside each demo cell below.

In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install numpy scikit-learn -q

# Reproducibility - the lesson uses seeded random values throughout
import numpy as np
np.random.seed(42)

**Explanation**

Pure environment prep. It fixes NumPy's global random seed so the make_classification / make_blobs / make_regression datasets generated later are identical every time you run the notebook.

**How the code works - step by step**
- **Commented `pip install`** - installs numpy and scikit-learn; left commented because Colab already has both.
- **`import numpy as np`** - the array library every sklearn dataset generator returns.
- **`np.random.seed(42)`** - pins the global RNG so seeded data is deterministic across runs.

**In one line:** import NumPy and lock the seed so every demo below is reproducible.

**What you'll see:** (no output - environment prep)

## 1 - Supervised learning: labels drive the loss

Supervised learning is the paradigm *with an answer key* — you have (X, y) pairs and the loss measures how far the prediction landed from the true y. Here we train a RandomForest on a synthetic labeled dataset and read off both its accuracy and which features it leaned on. This is the cat-dog-fish classifier idea at production scale: features in, label out, error scored against the truth.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X, y = make_classification(n_samples=2000, n_features=20,
                           n_informative=12, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestClassifier(100, random_state=42)
rf.fit(X_tr, y_tr)
print(f"Accuracy: {accuracy_score(y_te, rf.predict(X_te)):.4f}")
top = rf.feature_importances_.argsort()[-3:][::-1]
print(f"Top features: {top.tolist()}")
# Output:
# Accuracy: 0.9075
# Top features: [11, 2, 19]

**Explanation**

A minimal end-to-end supervised pipeline: generate labeled data, split it, fit a forest, score it on held-out data. The feature-importance line is the interesting extra — it shows the model didn't just predict, it ranked which of the 20 inputs actually mattered.

**How the code works - step by step**
- **`make_classification(...)`** - fabricates 2000 rows with 20 features, only 12 of them informative, plus a binary label y.
- **`train_test_split(...)`** - carves off 20% as a test set the model never sees during fitting.
- **`RandomForestClassifier(100).fit(...)`** - trains 100 decision trees on the training split.
- **`accuracy_score(y_te, rf.predict(X_te))`** - scores predictions on the held-out test set.
- **`rf.feature_importances_.argsort()[-3:][::-1]`** - pulls the indices of the 3 most influential features, highest first.

**In one line:** make labeled data -> split -> fit forest -> report test accuracy and the top-3 features.

**What you'll see:** Two printed lines: `Accuracy: 0.9075` and `Top features: [11, 2, 19]` — about 91% correct on unseen data, driven mainly by features 11, 2, and 19.

## 2 - Unsupervised learning: find structure with no labels

Strip the labels away and the question changes from *predict y* to *what groups does the data fall into on its own?* This cell runs KMeans for k = 2, 3, 4, 5 on blob data and scores each with the silhouette coefficient. The point is to watch two different signals — inertia and silhouette — disagree about what 'better' means.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score

X, _ = make_blobs(n_samples=300, centers=3, cluster_std=1.2, random_state=7)
for k in (2, 3, 4, 5):
    km = KMeans(n_clusters=k, n_init=10, random_state=7).fit(X)
    sil = silhouette_score(X, km.labels_)
    print(f"k={k}  inertia={km.inertia_:7.1f}  silhouette={sil:.3f}")
# Output:
# k=2  inertia= 3563.9  silhouette=0.728
# k=3  inertia=  825.7  silhouette=0.748
# k=4  inertia=  720.6  silhouette=0.620
# k=5  inertia=  625.6  silhouette=0.456

**Explanation**

A model-selection sweep, not a single fit. It clusters the same points at four different k values and prints two competing quality numbers per k, so you can see why inertia alone can't choose k but silhouette can.

**How the code works - step by step**
- **`make_blobs(centers=3, ...)`** - generates 300 points that genuinely come from 3 Gaussian blobs (the labels are discarded with `_`).
- **`for k in (2, 3, 4, 5)`** - tries four different cluster counts.
- **`KMeans(n_clusters=k, n_init=10).fit(X)`** - fits KMeans, restarting 10 times to avoid bad initializations.
- **`km.inertia_`** - within-cluster sum of squares; always shrinks as k grows.
- **`silhouette_score(X, km.labels_)`** - measures how tight-and-separated the clusters are; peaks at the true k.

**In one line:** cluster the same data at k=2..5 and print inertia (always falls) vs silhouette (peaks at the right answer).

**What you'll see:** Four lines of inertia and silhouette. Inertia drops monotonically (3563.9 -> 825.7 -> 720.6 -> 625.6) while silhouette peaks at `k=3` (0.748) — recovering the true 3-blob structure with no labels used.

## 3 - Generalization tool 1: regularization (Ridge vs Lasso)

Generalization is the only score that pays your salary — does the pattern survive data the model never trained on? This cell builds a hard case: 30 samples, 25 features, but only 4 of them real. Plain linear regression will memorize the noise; Ridge and Lasso each add a penalty on large weights, and they make opposite bets about what a good solution looks like.

In [ ]:
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split

# 30 samples, 25 features - but only 4 features actually matter
X, y = make_regression(n_samples=30, n_features=25, n_informative=4,
                       noise=25, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=1)

for name, model in [("Linear", LinearRegression()),
                    ("Ridge a=10", Ridge(alpha=10.0)),
                    ("Lasso a=2", Lasso(alpha=2.0, max_iter=50_000))]:
    model.fit(X_tr, y_tr)
    print(f"{name:11s} train R2={model.score(X_tr, y_tr):.3f}  "
          f"test R2={model.score(X_te, y_te):.3f}")
lasso = Lasso(alpha=2.0, max_iter=50_000).fit(X_tr, y_tr)
print(f"Lasso kept {(lasso.coef_ != 0).sum()} of 25 features")
# Output:
# Linear      train R2=1.000  test R2=0.606
# Ridge a=10  train R2=0.930  test R2=0.425
# Lasso a=2   train R2=0.993  test R2=0.861
# Lasso kept 15 of 25 features

**Explanation**

A controlled bake-off showing regularization as a *prior*, not a magic button. Same data, three models, printed as train-R2 next to test-R2 so the overfitting gap is visible — plus a count of how many features Lasso zeroed out entirely.

**How the code works - step by step**
- **`make_regression(n_features=25, n_informative=4, noise=25)`** - 30 rows, 25 features, only 4 that actually predict y, with heavy noise.
- **`train_test_split(test_size=0.4)`** - holds out 40% for honest testing.
- **`LinearRegression`** - no penalty; free to overfit.
- **`Ridge(alpha=10)`** - L2 penalty; shrinks every weight toward zero equally.
- **`Lasso(alpha=2)`** - L1 penalty; drives weak weights to exactly zero (feature selection).
- **`model.score(...)`** - prints R2 on train and test for each.
- **`(lasso.coef_ != 0).sum()`** - counts how many of the 25 features Lasso actually kept.

**In one line:** fit no-penalty / L2 / L1 on few-real-features data and compare train vs test R2.

**What you'll see:** Three train/test R2 lines plus a feature count. Linear overfits (train 1.000, test 0.606); Ridge's shrink-everything prior misfires here (test 0.425); Lasso wins (test 0.861) and reports `kept 15 of 25 features` — its zero-out bet matches the few-real-features truth.

## 4 - Generalization tool 2: k-fold cross-validation

Every R2 in the previous cell came from *one* train/test split — re-deal the cards and the numbers move. k-fold answers *how much* they move: split the data into 5 blocks, hold out a different block each round, retrain 5 times, and collect 5 verdicts instead of one. The spread across folds is the warning label a single split never gives you.

In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_score

for name, model in [("Ridge a=10", Ridge(alpha=10.0)),
                    ("Lasso a=2", Lasso(alpha=2.0, max_iter=50_000))]:
    scores = cross_val_score(model, X, y, cv=5)  # 5 retrains, 5 verdicts
    print(f"{name:11s} folds={np.round(scores, 2)}  "
          f"mean={scores.mean():.3f}  spread={scores.max()-scores.min():.3f}")
# Output:
# Ridge a=10  folds=[ 0.94 -0.12  0.55  0.62  0.65]  mean=0.529  spread=1.068
# Lasso a=2   folds=[0.92 0.92 0.88 0.96 0.95]  mean=0.926  spread=0.074

**Explanation**

A stability measurement, not a new model. It runs 5-fold cross-validation on the same Ridge and Lasso from the previous cell and reports each fold's score, the mean, and the max-minus-min spread — so a wide spread flags an estimate you can't trust.

**How the code works - step by step**
- **reuses `X, y`** - the same 30-sample regression data from cell 3.
- **`cross_val_score(model, X, y, cv=5)`** - retrains the model 5 times, each on 4/5 of the data, scoring on the held-out fifth.
- **`np.round(scores, 2)`** - shows the 5 individual fold scores.
- **`scores.mean()`** - the trustworthy point estimate across folds.
- **`scores.max() - scores.min()`** - the spread; a big spread means the single-split number was luck.

**In one line:** retrain each model on 5 different splits and report the fold scores, their mean, and their spread.

**What you'll see:** Two lines. Ridge is unstable — folds swing from 0.94 to -0.12 (mean 0.529, spread 1.068); Lasso is tight — folds all near 0.9 (mean 0.926, spread 0.074). The spread is what tells you Ridge's estimate here is mush and Lasso's is real.

Four small sklearn demos carried the whole lesson: RandomForest showed supervised learning (labels drive the loss), KMeans showed unsupervised structure (silhouette picks k with no labels), and the Ridge/Lasso + k-fold pair showed generalization as a measurable thing — Lasso's zero-out prior beating Ridge on few-real-features data, and 5-fold exposing Ridge's 0.94-vs--0.12 instability that one lucky split would have hidden. The through-line is that the loss function, not the architecture, decides which pattern gets learned, and validation — not training score — decides whether it survives. Lesson 1.6 rebuilds KMeans from scratch, and Module 14's golden eval sets are this same generalization discipline at LLM scale.